# Lab 1 — K-Means Baseline

**Day 05 · Unsupervised Learning · Cisco AI/ML Training**

---

## Goals

1. Build symbol-level features from NYSE daily candles.
2. Standardize numeric features before clustering.
3. Fit K-Means with a baseline `k=4` and inspect inertia.
4. Read cluster sizes to understand segment balance.

**Dataset:** `data/nyse/nyse_stocks.csv` (500 rows, 25 symbols)

## Baseline workflow

Daily rows -> aggregate to symbol features -> scale -> K-Means -> inspect counts and separation.

In [ ]:
%matplotlib inline

from pathlib import Path

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

FEATURE_COLUMNS = ["avg_close", "volatility", "avg_volume", "avg_range"]

nyse = pd.read_csv(GH_ROOT / "data" / "nyse" / "nyse_stocks.csv", parse_dates=["date"])
print(f"dataset path: {GH_ROOT / 'data' / 'nyse' / 'nyse_stocks.csv'}")
print(f"daily rows: {len(nyse)}")
print(f"symbols: {nyse['symbol'].nunique()}")
display(nyse.head(3))


## Build symbol features

In [ ]:

nyse["range"] = nyse["high"] - nyse["low"]
features = (
    nyse.groupby("symbol")
    .agg(
        avg_close=("close", "mean"),
        volatility=("close", "std"),
        avg_volume=("volume", "mean"),
        avg_range=("range", "mean"),
    )
    .reset_index()
)
features["volatility"] = features["volatility"].fillna(0.0)

print(f"symbols clustered: {len(features)}")
display(features.head(5))


## Inspect descriptive statistics

In [ ]:
display(features[FEATURE_COLUMNS].describe().round(3))
print("Any missing values by column:")
print(features[FEATURE_COLUMNS].isna().sum())


## Prepare matrix and scale

In [ ]:
X = features[FEATURE_COLUMNS]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_df = pd.DataFrame(X_scaled, columns=FEATURE_COLUMNS)
display(pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "raw": X.iloc[0].round(2).values,
    "scaled": scaled_df.iloc[0].round(2).values,
}))


## Fit baseline K-Means

In [ ]:
k = 4
model = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = model.fit_predict(X_scaled)

features = features.copy()
features["cluster"] = labels

print("Lab 1 — K-Means baseline")
print(f"features: {FEATURE_COLUMNS}")
print(f"k: {k}")
print(f"inertia: {model.inertia_:.4f}")

unique, counts = np.unique(labels, return_counts=True)
cluster_counts = dict(zip(unique.tolist(), counts.tolist()))
print(f"cluster counts: {cluster_counts}")


## Cluster centers in scaled space

In [ ]:
centers_scaled = pd.DataFrame(model.cluster_centers_, columns=FEATURE_COLUMNS)
display(centers_scaled.round(3))


## Cluster centers in original units

In [ ]:
centers_raw = pd.DataFrame(
    scaler.inverse_transform(model.cluster_centers_),
    columns=FEATURE_COLUMNS,
)
display(centers_raw.round(2))


## Sorted cluster membership

In [ ]:
display(
    features[["symbol", "avg_close", "volatility", "cluster"]]
    .sort_values(["cluster", "symbol"])
    .round(2)
)


## Visual cluster scatter

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=features,
    x="avg_close",
    y="volatility",
    hue="cluster",
    palette="tab10",
    s=100,
    ax=ax,
)
for _, row in features.iterrows():
    ax.annotate(row["symbol"], (row["avg_close"], row["volatility"]), fontsize=8, alpha=0.8)
ax.set_title(f"K-Means clusters (k={k}, inertia={model.inertia_:.1f})")
plt.tight_layout()
plt.show()


## Compare with k=3

In [ ]:
model_k3 = KMeans(n_clusters=3, random_state=42, n_init=10)
model_k3.fit_predict(X_scaled)

compare_k = pd.DataFrame({
    "k": [3, 4],
    "inertia": [model_k3.inertia_, model.inertia_],
})
display(compare_k.round(4))
print("Lower inertia with more clusters — Lab 2 finds the best k via the elbow method.")


### Guided mini-check 1

How many symbols land in each cluster when sorted by label?

In [ ]:
print(features['cluster'].value_counts().sort_index())

### Guided mini-check 2

Which symbol has the highest average close?

In [ ]:
display(features.nlargest(3, 'avg_close')[['symbol', 'avg_close', 'cluster']].round(2))

### Guided mini-check 3

Which symbol has the highest volatility?

In [ ]:
display(features.nlargest(3, 'volatility')[['symbol', 'volatility', 'cluster']].round(3))

### Guided mini-check 4

Do scaled features have mean ~0?

In [ ]:
print(pd.DataFrame(X_scaled, columns=FEATURE_COLUMNS).mean().round(6))

### Guided mini-check 5

Do scaled features have std ~1?

In [ ]:
print(pd.DataFrame(X_scaled, columns=FEATURE_COLUMNS).std(ddof=0).round(6))

### Guided mini-check 6

Review within-cluster average features.

In [ ]:
display(features.groupby('cluster')[FEATURE_COLUMNS].mean().round(2))

### Guided mini-check 7

Compare cluster median volatility.

In [ ]:
display(features.groupby('cluster')['volatility'].median().round(3))

### Guided mini-check 8

Check symbol order within one cluster.

In [ ]:
display(features.loc[features['cluster']==0, ['symbol','avg_close']].sort_values('symbol'))

### Guided mini-check 9

Inspect farthest symbols by avg_range.

In [ ]:
display(features.nlargest(5, 'avg_range')[['symbol','avg_range','cluster']].round(2))

### Guided mini-check 10

Summarize inertia and cluster count.

In [ ]:
print({'k': k, 'inertia': round(model.inertia_, 4), 'clusters': len(cluster_counts)})

### Guided mini-check 11

What fraction of symbols are in cluster 1?

In [ ]:
print(round((features['cluster']==1).mean(), 3))

### Guided mini-check 12

Show top-2 avg_volume per cluster.

In [ ]:
display(features.sort_values(['cluster','avg_volume'], ascending=[True, False]).groupby('cluster').head(2)[['symbol','cluster','avg_volume']].round(2))

### Guided mini-check 13

Cross-check feature matrix shape.

In [ ]:
print('X shape:', X.shape, '| X_scaled shape:', X_scaled.shape)

### Guided mini-check 14

Compare mean avg_close by cluster.

In [ ]:
print(features.groupby('cluster')['avg_close'].mean().round(2).to_dict())

### Guided mini-check 15

Inspect one symbol end-to-end row.

In [ ]:
display(features.loc[features['symbol']=='MSFT', ['symbol', *FEATURE_COLUMNS, 'cluster']].round(2))

### Guided mini-check 16

Check whether any two symbols share identical feature rows.

In [ ]:
print(features[FEATURE_COLUMNS].duplicated().sum())

### Guided mini-check 17

List symbols per cluster quickly.

In [ ]:
print(features.groupby('cluster')['symbol'].apply(list).to_dict())

### Guided mini-check 18

Sort by volatility and review first 8 rows.

In [ ]:
display(features[['symbol','volatility','cluster']].sort_values('volatility').head(8).round(3))

### Guided mini-check 19

Verify no null in model input.

In [ ]:
print(pd.DataFrame(X_scaled, columns=FEATURE_COLUMNS).isna().sum().to_dict())

### Guided mini-check 20

Re-state baseline takeaway in code output.

In [ ]:
print('Baseline k=4 creates four compact groups with one singleton cluster.')

## Final checkpoint

In [ ]:
assert len(features) == 25
assert k == 4
assert abs(model.inertia_ - 45.8634) < 0.1
assert cluster_counts == {0: 8, 1: 9, 2: 7, 3: 1}
print("Numbers match — you're good.")



## Reflection

How does changing `k` affect inertia and interpretability trade-offs?